# Chains

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_community.document_loaders import TextLoader,PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma,FAISS
import fitz  

#utility
import numpy as np
from typing import List,Dict,Any
from sentence_transformers import SentenceTransformer

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import (RunnablePassthrough,RunnableMap)
from langchain_core.output_parsers import StrOutputParser
import os
import numpy as np
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.document_loaders import WikipediaLoader
from PIL import Image
import torch
from transformers import CLIPProcessor,CLIPModel
import base64
import io
from langchain.messages import SystemMessage,HumanMessage,AIMessage
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware,HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
from langchain.tools import tool
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from IPython.display import Image,display

## Reducers
from typing import Annotated
from typing import Literal
from langgraph.graph.message import add_messages
import random
from  dataclasses import dataclass
from pydantic import BaseModel
from pprint import pprint


C:\Users\kanha\AppData\Local\Temp\ipykernel_28064\3962715687.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader,PyMuPDFLoader


In [2]:
## Step 4: LLM and prompt
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"] =os.getenv("OPENAI_API_KEY")
llm=init_chat_model(model="groq:llama-3.3-70b-versatile")
response=llm.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots are known for their remarkable ability to mimic human speech and other sounds, but why do they do it? The answer lies in their evolution, social behavior, and communication needs.\n\n**Mimicry as a form of communication**\n\nIn the wild, parrots use vocalizations to communicate with each other. They make various sounds to convey information, warnings, and emotions, such as:\n\n1. Contact calls: to maintain contact with flock members or attract a mate.\n2. Alarm calls: to warn others of potential predators.\n3. Food calls: to signal the discovery of food sources.\n\nMimicry is an extension of this communication system. By imitating other sounds, including human speech, parrots can:\n\n1. **Imitate predators**: to warn other parrots of potential threats.\n2. **Attract a mate**: by demonstrating their vocal abilities and intelligence.\n3. **Establish dominance**: by mimicking the calls of other birds or even other animals.\n\n**Social learning and bonding**\n\nP

In [6]:
from pprint import pprint
messages=[
    AIMessage(content=f"Please tell me how can i help",name="LLM")
]
messages.append(HumanMessage(content=f"I want to learn coding",name="Saksham"))
messages.append(
    AIMessage(content=f"Which language you want to learn",name="LLM")
)
messages.append(HumanMessage(content=f"I want to learn python",name="Saksham"))

for message in messages:
    message.pretty_print()

================================== Ai Message ==================================
Name: LLM

Please tell me how can i help
================================ Human Message =================================
Name: Saksham

I want to learn coding
================================== Ai Message ==================================
Name: LLM

Which language you want to learn
================================ Human Message =================================
Name: Saksham

I want to learn python


In [8]:
response=llm.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================

Python is a great language to start with. It's easy to learn, versatile, and has a large community of developers who contribute to its ecosystem.

To get started with Python, here are some steps you can follow:

1. **Install Python**: Download and install the latest version of Python from the official Python website: <https://www.python.org/downloads/>
2. **Choose a text editor or IDE**: A text editor or Integrated Development Environment (IDE) is where you'll write your Python code. Some popular choices include:
	* PyCharm
	* Visual Studio Code (VS Code)
	* Sublime Text
	* Atom
3. **Learn the basics**: Start with basic syntax and data types, such as:
	* Variables
	* Data types (strings, lists, dictionaries, etc.)
	* Control structures (if-else statements, loops, etc.)
	* Functions
4. **Practice, practice, practice**: Practice writing Python code to solidify your understanding of the concepts. You can sta

In [14]:
response.response_metadata

{'token_usage': {'completion_tokens': 483,
  'prompt_tokens': 73,
  'total_tokens': 556,
  'completion_time': 0.970488575,
  'completion_tokens_details': None,
  'prompt_time': 0.00664735,
  'prompt_tokens_details': None,
  'queue_time': 0.162033388,
  'total_time': 0.977135925},
 'model_name': 'llama-3.3-70b-versatile',
 'system_fingerprint': 'fp_3272ea2d91',
 'service_tier': 'on_demand',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'groq'}